# Activity: Policy Iteration for Inventory Control
Inventory control is a classic sequential decision problem: a manager observes on-hand stock, decides how much to order, and then sells against random demand while paying for ordering, holding excess stock, and lost sales when demand exceeds supply. In this activity, you will build the MDP tuple $(\mathcal{S},\mathcal{A},T,R,\gamma)$ for a single-item inventory system with stochastic demand, then implement policy iteration yourself, sweep by sweep, alternating exact policy evaluation with greedy policy improvement. You will then count how many sweeps value iteration needs to reach the same optimal policy, and compare the two counts.

> __Learning Objectives.__
>
> In this activity, you will build a stochastic inventory MDP, implement policy iteration step by step, and compare its convergence to value iteration:
> * __Build the inventory MDP:__ construct the tuple $(\mathcal{S},\mathcal{A},T,R,\gamma)$ for a single-item inventory system with `build_inventory_mdp`.
> * __Implement policy iteration step by step:__ alternate exact policy evaluation with `policy_evaluation` and greedy improvement with `policy(Q(...))`, tracking how many actions change on each sweep until the policy stabilizes.
> * __Compare its iteration count to value iteration:__ count how many value-iteration sweeps are needed to first reach the same optimal policy, and contrast that with the number of policy-iteration sweeps.

Let's get started.
___

## Theory
For a finite MDP $(\mathcal{S},\mathcal{A},P,R,\gamma)$ with states $s\in\mathcal{S}$, actions $a\in\mathcal{A}$, transition probabilities $P(s^{\prime}\mid s,a)$, expected immediate reward $R(s,a)$, discount factor $\gamma\in[0,1)$, and a deterministic policy $\pi:\mathcal{S}\rightarrow\mathcal{A}$, the state-value function $V^{\pi}$ satisfies the __Bellman expectation equation__:
$$
V^{\pi}(s) = R(s,\pi(s)) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,\pi(s))\,V^{\pi}(s^{\prime})\quad\forall s\in\mathcal{S}.
$$
Writing it in matrix form with the value vector $\mathbf{V}^{\pi}\in\mathbb{R}^{|\mathcal{S}|}$, the reward vector $\mathbf{R}^{\pi}$ with entries $R(s,\pi(s))$, and the $|\mathcal{S}|\times|\mathcal{S}|$ transition matrix $\mathbf{P}^{\pi}$ with entries $P(s^{\prime}\mid s,\pi(s))$:
$$
\mathbf{V}^{\pi} = \mathbf{R}^{\pi} + \gamma\,\mathbf{P}^{\pi}\mathbf{V}^{\pi}
\quad\Longrightarrow\quad
\mathbf{V}^{\pi} = \left(\mathbf{I} - \gamma\,\mathbf{P}^{\pi}\right)^{-1}\mathbf{R}^{\pi}.
$$
The matrix $\mathbf{I}-\gamma\mathbf{P}^{\pi}$ is invertible because $\gamma<1$ and $\mathbf{P}^{\pi}$ is a stochastic matrix, so this __exact policy evaluation__ step is a single linear solve; the `policy_evaluation` function you will call performs it. Given $V^{\pi}$, __policy improvement__ defines a new policy that is greedy with respect to the one-step lookahead, i.e., the action-value function $Q^{\pi}(s,a) = R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V^{\pi}(s^{\prime})$:
$$
\pi^{\prime}(s) = \arg\max_{a\in\mathcal{A}} Q^{\pi}(s,a)\quad\forall s\in\mathcal{S}.
$$
The __policy improvement theorem__ guarantees $V^{\pi^{\prime}}(s)\ge V^{\pi}(s)$ for all $s\in\mathcal{S}$, with equality only when $\pi$ is already optimal. __Policy iteration__ starts from an arbitrary policy $\pi_{0}$ and repeats evaluation and improvement, $\pi_{k+1} = \text{improve}\left(\text{evaluate}(\pi_{k})\right)$, until $\pi_{k+1}=\pi_{k}$. A finite MDP has finitely many policies, and each iteration strictly improves the policy until none is better, so policy iteration converges to an optimal policy $\pi^{\star}$ in a finite number of sweeps, each of which costs one exact linear solve plus one greedy maximization.

For comparison, __value iteration__ applies the Bellman optimality operator
$$
(BV)(s) = \max_{a\in\mathcal{A}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V(s^{\prime})\right)\quad\forall s\in\mathcal{S}
$$
repeatedly, $V_{k+1}=BV_{k}$, starting from $V_{0}(s)=0$ for all $s\in\mathcal{S}$. Because $B$ is a contraction in the sup-norm $\lVert\cdot\rVert_{\infty}$ with modulus $\gamma$, $V_{k}\rightarrow V^{\star}$ as $k\rightarrow\infty$, but the greedy policy $\pi_{k}(s)=\arg\max_{a}Q_{k}(s,a)$ recovered from $V_{k}$ can match $\pi^{\star}$ well before $V_{k}$ itself has numerically converged. This activity compares how many sweeps each algorithm needs to reach $\pi^{\star}$: policy iteration counts full evaluate-improve cycles, while value iteration counts single backups of $(BV)(s)$.
___

## Setup
This activity uses functions defined in the `src` directory and a small set of external packages. The `include(...)` call below runs `Include.jl`, which activates the local project environment, loads the packages, and includes our code. The first run may take a few minutes while packages are installed and precompiled.

In [1]:
include("Include.jl");

  Activating 

project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-145/module-4`


## Build the Inventory MDP
`build_inventory_mdp` constructs the MDP tuple $(\mathcal{S},\mathcal{A},T,R,\gamma)$ for a single-item inventory system. State $s=i+1$ encodes on-hand inventory $i\in\{0,\dots,\text{capacity}\}$; action $a=o+1$ encodes an order quantity $o\in\{0,\dots,\text{capacity}\}$. After ordering, post-order stock is $q=i+o$; an order that pushes $q$ above `capacity` is infeasible and self-loops with a large penalty reward. Demand $d$ is drawn from `demand_pmf` over $\{0,1,2,3,4\}$; sales are $\min(q,d)$, unmet demand is $\max(0,d-q)$, and next on-hand is $\max(0,q-d)$. The expected reward nets sales revenue at `price` against `order_cost` per unit ordered, a `fixed_cost` charged whenever $o>0$, `holding_cost` per unit of post-order stock, and a `stockout_penalty` per unit of unmet demand.

We build the same capacity-8 system used in the Watch-Demo, with a symmetric, unimodal demand distribution centered at $d=2$.

In [2]:
mdp = build_inventory_mdp(; capacity = 8, demand_pmf = [0.05, 0.2, 0.5, 0.2, 0.05],
    price = 10.0, order_cost = 3.0, fixed_cost = 2.0, holding_cost = 1.0,
    stockout_penalty = 8.0, γ = 0.95);
(length(mdp.𝒮), length(mdp.𝒜))

(9, 9)

## Step Through Policy Iteration Manually
Implement the policy-iteration loop yourself. Starting from $\pi_{0}(s)=1$ at every state (action index $1\Rightarrow$ order nothing), each sweep $k$: (i) evaluates the current policy exactly with `policy_evaluation(mdp, π)`, which solves $\mathbf{V}^{\pi}=(\mathbf{I}-\gamma\mathbf{P}^{\pi})^{-1}\mathbf{R}^{\pi}$; (ii) improves it greedily with `policy(Q(mdp, V))`, which returns $\pi^{\prime}(s)=\arg\max_{a}Q^{\pi}(s,a)$; and (iii) reports how many state actions changed between $\pi$ and $\pi^{\prime}$. The loop stops as soon as $\pi^{\prime}=\pi$, i.e., no action changed on that sweep, which is exactly the policy iteration stopping criterion from the Theory section above.

In [3]:
let
    π = ones(Int64, length(mdp.𝒮));   # start: order 0 everywhere
    for k ∈ 1:20
        V = policy_evaluation(mdp, π);
        π′ = policy(Q(mdp, V));
        println("sweep $(k):  changed actions = ", count(π′ .!= π));
        (π′ == π) && (println("converged after $(k) sweep(s)"); break);
        π = π′;
    end
end

sweep 1:  changed actions = 8


sweep 2:  changed actions = 8
sweep 3:  changed actions = 0
converged after 3 sweep(s)


## Count Value-Iteration Sweeps to the Same Policy
Now find out how many value-iteration sweeps are needed to reach the same optimal policy $\pi^{\star}$ that policy iteration converged to above. We first recover $\pi^{\star}$ from a fresh `solve(build(MyPolicyIterationModel, ...), mdp)` call, then run the raw Bellman optimality backup $V_{k+1}=BV_{k}$ starting from $V_{0}=\mathbf{0}$, one sweep at a time, and record the first sweep $k$ at which the greedy policy $\pi_{k}=\text{policy}(Q(\text{mdp}, V_{k}))$ equals $\pi^{\star}$. Each value-iteration sweep is a single vectorized backup over every state and action — much cheaper per sweep than a policy-iteration sweep, which requires a full linear solve — but, as you will see, value iteration needs more of them to match $\pi^{\star}$.

In [4]:
let
    π_star = policy(Q(mdp, solve(build(MyPolicyIterationModel, (maxiterations = 100,)), mdp).V));
    V = zeros(Float64, length(mdp.𝒮));
    k = 0; matched = -1;
    while (k < 5_000)
        V = [ maximum(mdp.R[s,a] + mdp.γ*sum(mdp.T[s,s′,a]*V[s′] for s′ ∈ mdp.𝒮) for a ∈ mdp.𝒜) for s ∈ mdp.𝒮 ];
        k += 1;
        if (policy(Q(mdp, V)) == π_star && matched < 0)
            matched = k;
        end
    end
    println("value iteration first matched the optimal policy at sweep ", matched);
end

value iteration first matched the optimal policy at sweep 2

## Experiment
Try changing `stockout_penalty` or `demand_pmf` in the cell below, rebuild the MDP with `build_inventory_mdp`, re-solve with policy iteration, and compare the resulting order-up-to level $S(i) = i + \pi(i)$ to the baseline solved with the original parameters above. Raising `stockout_penalty` makes stockouts costlier, so the optimal policy should order up to a higher level to buffer against demand variability; reshaping `demand_pmf` toward higher demand should do the same.
>__Note:__ if you change `demand_pmf`, it must remain a valid probability distribution — all entries non-negative and summing to $1$ — otherwise the rows of the transition array $T$ will not sum to $1$ and the solved policy will be meaningless.

In [5]:
let
    sol_base = solve(build(MyPolicyIterationModel, (maxiterations = 100,)), mdp);
    π_base = policy(Q(mdp, sol_base.V));
    S_base = [ i + (π_base[i+1] - 1) for i ∈ 0:8 ];

    stockout_try = 20.0;   # try changing this (or demand_pmf below, keeping it a valid pmf)
    mdp_try = build_inventory_mdp(; capacity = 8, demand_pmf = [0.05, 0.2, 0.5, 0.2, 0.05],
        price = 10.0, order_cost = 3.0, fixed_cost = 2.0, holding_cost = 1.0,
        stockout_penalty = stockout_try, γ = 0.95);
    sol_try = solve(build(MyPolicyIterationModel, (maxiterations = 100,)), mdp_try);
    π_try = policy(Q(mdp_try, sol_try.V));
    S_try = [ i + (π_try[i+1] - 1) for i ∈ 0:8 ];

    println("baseline (stockout_penalty = 8.0):    order-up-to S = ", S_base);
    println("stockout_penalty = $(stockout_try):   order-up-to S = ", S_try);
end

baseline (stockout_penalty = 8.0):    order-up-to S = 

[3, 3, 3, 3, 4, 5, 6, 7, 8]
stockout_penalty = 20.0:   order-up-to S = [4, 4, 4, 3, 4, 5, 6, 7, 8]


## Summary
This activity built a capacity-8 single-item inventory MDP with stochastic demand, implemented policy iteration sweep by sweep, and counted how many value-iteration sweeps were needed to reach the same optimal policy.

Policy iteration reaches the optimal policy in a small number of evaluate-improve sweeps, each of which is expensive because it solves a linear system exactly; value iteration reaches the same policy only after many cheap Bellman-backup sweeps.

> __Key Takeaways:__
>
> * **Policy iteration converges in a few expensive sweeps:** each sweep couples one exact linear solve, `policy_evaluation(mdp, π)`, with one greedy improvement, `policy(Q(mdp, V))`, and the loop above converges in a handful of sweeps.
> * **Value iteration takes more cheap sweeps to reach the same policy:** each value-iteration sweep is a single vectorized Bellman optimality backup, cheaper than a policy-iteration sweep, but many more of them are needed before the greedy policy first matches $\pi^{\star}$.
> * **Both agree at the optimum:** policy iteration and value iteration solve the same Bellman optimality equation by different routes, and both converge to the same optimal policy $\pi^{\star}$ on this inventory MDP.

Policy iteration and value iteration are two routes to the same fixed point $\pi^{\star}$, trading the number of sweeps against the cost per sweep. When the transition and reward model is fully known and the state space is small enough for the linear solve to be cheap, policy iteration's exactness lets it reach $\pi^{\star}$ in far fewer sweeps than value iteration.
___

### Additional Resources
* Puterman, M. L. (1994). _Markov Decision Processes: Discrete Stochastic Dynamic Programming_. John Wiley & Sons.
* Bertsekas, D. P. (2017). _Dynamic Programming and Optimal Control_ (4th ed.). Athena Scientific.
* Sutton, R. S., & Barto, A. G. (2018). _Reinforcement Learning: An Introduction_ (2nd ed.). MIT Press.